<h1 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">مشروع الرؤية الحاسوبية والتعلّم العميق: الفرز الذكي للنفايات</h1>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext"><strong>الفكرة:</strong> تحليل صورة جسم مرمي وتصنيفه إلى زجاج، ورق، كرتون، بلاستيك، معدن، أو نفايات أخرى. نستخدم OpenCV لفحص الصورة وتحسينها، ثم شبكة CNN للتعلّم والتصنيف.</p>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext"><strong>سؤال المشروع:</strong> هل يستطيع النموذج تعميم ما تعلمه من صور بخلفيات بسيطة على صور جديدة التقطناها في بيئتنا؟</p>
<blockquote dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">هذا نموذج تعليمي للتصنيف بالصورة، وليس نظامًا آليًا يتحكم بمعدات فرز حقيقية.</blockquote>

<h2 dir="rtl" style="direction:rtl;text-align:right">خارطة التنفيذ</h2>
<ol dir="rtl" style="direction:rtl;text-align:right"><li>تهيئة بيئة Colab والتحقق من GPU، ثم رفع ملف الدوال المشتركة عند الطلب.</li><li>تنزيل TrashNet وفحص الفئات والأعداد وجودة الصور.</li><li>قراءة الصورة بـ OpenCV ومقارنة RGB وHSV وCLAHE، ثم عرض سطح شدة الحواف ثلاثي الأبعاد.</li><li>تقسيم طبقي إلى تدريب وتحقق واختبار، وتجهيز دوال التحميل والدفعات وأوزان الفئات.</li><li>إنشاء CNN صغيرة، وفهم شكل الموترات، ثم تتبع التدريب.</li><li>تقييم مستقل بمقاييس لكل فئة ومصفوفة الالتباس.</li><li>عرض خرائط الخصائص ثلاثية الأبعاد، ثم تجربة واجهة رفع الصورة.</li></ol>
<p dir="rtl" style="direction:rtl;text-align:right"><strong>طريقة الاستخدام:</strong> شغّل الخلايا من أعلى إلى أسفل. إذا انقطع Colab أو أُعيد تشغيل الجلسة، أعد تشغيل الخلايا بالترتيب؛ ذاكرة المتغيرات لا تبقى بعد إعادة الاتصال.</p>

In [ ]:
# هذا الدفتر مُعدّ للتشغيل في Google Colab.
import sys, os, random, zipfile, urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:
# الدوال المشتركة مع واجهة الويب المحلية.
# عند العمل في Colab، اختر waste_sorting_helpers.py من جهازك عند ظهور نافذة الرفع.
import importlib.util
try:
    import waste_sorting_helpers as wsh
except ModuleNotFoundError:
    from google.colab import files
    uploaded_helpers = files.upload()
    helper_name = next((name for name in uploaded_helpers if name.endswith("waste_sorting_helpers.py")), None)
    if helper_name is None:
        raise FileNotFoundError("ارفع ملف waste_sorting_helpers.py المرفق مع المشروع.")
    spec = importlib.util.spec_from_file_location("waste_sorting_helpers", helper_name)
    wsh = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = wsh
    spec.loader.exec_module(wsh)
print("تم تحميل الدوال المشتركة:", wsh.__file__)


In [ ]:
%pip -q install arabic-reshaper python-bidi plotly
import arabic_reshaper
from bidi.algorithm import get_display

def rtl_text(text):
    """تهيئة النص العربي ليظهر باتجاه صحيح في رسوم Matplotlib."""
    return get_display(arabic_reshaper.reshape(str(text)))

print("تم تجهيز أدوات تشكيل العربية والرسوم التفاعلية ثلاثية الأبعاد.")

In [ ]:
# تنزيل النسخة المصغّرة (حوالي 41 MB) وفكها تلقائيًا داخل Colab.
DATA_ROOT = Path("/content/trashnet")
DATA_DIR = DATA_ROOT / "dataset-resized"
ZIP_PATH = Path("/content/trashnet_dataset.zip")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    url = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"
    urllib.request.urlretrieve(url, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)

if not DATA_DIR.exists():
    # بعض نسخ الأرشيف تضع مجلد الفئات مباشرة داخل مجلد آخر.
    candidates = [p for p in DATA_ROOT.rglob("*") if p.is_dir() and any(x.is_dir() for x in p.iterdir())]
    DATA_DIR = next((p for p in candidates if {"glass", "paper"}.issubset({x.name.lower() for x in p.iterdir() if x.is_dir()})), DATA_DIR)
print("مجلد البيانات:", DATA_DIR)


In [ ]:
# جرد الصور والفئات باستخدام الدالة المشتركة مع تطبيق الويب.
dataset_info = wsh.scan_dataset(DATA_DIR)
class_names = dataset_info["classes"]
records = [(str(path), class_names.index(label)) for path, label in dataset_info["records"]]
paths = np.array([row[0] for row in records])
labels = np.array([row[1] for row in records], dtype=np.int32)
print("مجلد الفئات:", dataset_info["root"])
print("الفئات:", class_names)
print("إجمالي الصور الصالحة:", dataset_info["total_images"])
print("صور غير قابلة للقراءة:", dataset_info["unreadable_images"])
for name in class_names:
    print(f"{name:12s}: {dataset_info['counts'][name]}")


<h2 dir="rtl" style="direction:rtl;text-align:right">فهم مجموعة البيانات وتوازن الفئات</h2>
<p dir="rtl" style="direction:rtl;text-align:right">كل مجلد يمثل فئة، وكل صورة داخله مثال تدريبي موسوم باسم المجلد. نعد الصور قبل التدريب حتى نكشف عدم توازن البيانات؛ فالفئة الأقل قد يتعلمها النموذج بصعوبة. هذا الرسم وصف للبيانات فقط ولا يثبت جودة التصنيف.</p>

In [ ]:
# دالة مساعدة: حساب عدد الملفات الصورية في كل فئة ورسمها.
from collections import Counter
class_counts = Counter(labels.tolist())
fig, ax = plt.subplots(figsize=(9, 4))
keys = list(range(len(class_names)))
ax.bar([class_names[k] for k in keys], [class_counts[k] for k in keys], color="#168aad")
ax.set_title(rtl_text("توزيع الصور حسب الفئة")); ax.set_ylabel("عدد الصور")
ax.tick_params(axis="x", rotation=25); ax.grid(axis="y", alpha=.2)
plt.tight_layout(); plt.show()
print("الفئة الأقل عددًا:", class_names[min(class_counts, key=class_counts.get)], "—", min(class_counts.values()), "صورة")

<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">فحص ومعالجة الصورة بـ OpenCV</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">صور OpenCV الملونة تقرأ افتراضيًا بترتيب BGR. نعرض تحويلها إلى RGB، ونفحص تمثيل HSV، ثم نجرّب CLAHE لتحسين التباين المحلي. في التدريب سنحافظ على RGB ونستخدم تحجيمًا وزيادة بيانات؛ اعرض تأثير CLAHE وناقش إن كان مفيدًا قبل اعتماد أي تحسين في النموذج.</p>

In [ ]:
sample_path = paths[0]
bgr = cv2.imread(sample_path)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
hue = hsv[:, :, 0]
lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(rgb); ax[0].set_title(rtl_text("RGB الأصلي"))
ax[1].imshow(hue, cmap="hsv"); ax[1].set_title(rtl_text("قناة Hue من HSV"))
ax[2].imshow(enhanced); ax[2].set_title(rtl_text("بعد CLAHE"))
for a0 in ax: a0.axis("off")
plt.tight_layout(); plt.show()


<h3 dir="rtl" style="direction:rtl;text-align:right">محاكاة سطح الحواف ثلاثي الأبعاد</h3>
<p dir="rtl" style="direction:rtl;text-align:right">نحوّل الصورة إلى رمادي، ثم نحسب مقدار التغير المحلي باستخدام Sobel. ارتفاع السطح يرمز إلى قوة الحافة؛ القمم لا تعني شقوقًا أو عيوبًا، بل انتقالات لونية/شكلية في صورة النفاية. هذا عرض بصري تعليمي لنتيجة معالجة OpenCV.</p>

In [ ]:
# دالة مساعدة لعرض شدة الحواف كمساحة ثلاثية الأبعاد.
import plotly.graph_objects as go

def show_edge_surface(image_rgb, max_side=90):
    """يعرض مقدار تدرج Sobel على شبكة مخفّضة حتى يبقى الرسم سريعًا في Colab."""
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    scale = min(max_side / h, max_side / w, 1.0)
    small = cv2.resize(gray, (max(8, int(w*scale)), max(8, int(h*scale))))
    gx = cv2.Sobel(small, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(small, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = cv2.magnitude(gx, gy)
    fig = go.Figure(data=[go.Surface(z=magnitude, colorscale="Viridis", showscale=True)])
    fig.update_layout(title="OpenCV Sobel — سطح مقدار الحواف", scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Edge strength"), height=520)
    fig.show()

show_edge_surface(rgb)

<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تقسيم البيانات وتجهيز الدفعات</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">نفصل مجموعة الاختبار قبل التدريب، ولا نستخدمها لاختيار النموذج. التقسيم طبقي ليحافظ على نسبة كل فئة، ونحسب أوزان الفئات لأن عدد صور «trash» أقل من بعض الفئات الأخرى.</p>

In [ ]:
# تقسيم طبقي وتجهيز tf.data من الدوال المساعدة المشتركة.
IMG_SIZE = (160, 160)
IMAGE_SIZE = IMG_SIZE[0]
BATCH_SIZE = 16
records_for_split = [(Path(path), class_names[int(label)]) for path, label in zip(paths, labels)]
train_records, val_records, test_records = wsh.stratified_split(
    records_for_split, test_percent=15, validation_percent=15, seed=SEED)
class_to_id = {name: index for index, name in enumerate(class_names)}
train_paths = np.array([str(path) for path, _ in train_records])
val_paths = np.array([str(path) for path, _ in val_records])
test_paths = np.array([str(path) for path, _ in test_records])
train_y = np.array([class_to_id[label] for _, label in train_records], dtype=np.int32)
val_y = np.array([class_to_id[label] for _, label in val_records], dtype=np.int32)
test_y = np.array([class_to_id[label] for _, label in test_records], dtype=np.int32)
train_ds = wsh.build_tf_dataset(tf, train_records, class_names, IMAGE_SIZE, BATCH_SIZE, training=True, seed=SEED)
val_ds = wsh.build_tf_dataset(tf, val_records, class_names, IMAGE_SIZE, BATCH_SIZE)
test_ds = wsh.build_tf_dataset(tf, test_records, class_names, IMAGE_SIZE, BATCH_SIZE)
class_weights = wsh.make_class_weights(class_names, train_records)
print(f"تدريب: {len(train_y)} | تحقق: {len(val_y)} | اختبار: {len(test_y)}")
print("أوزان الفئات:", {class_names[k]: round(v, 2) for k, v in class_weights.items()})


In [ ]:
# معاينة عينات بعد التحميل
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, (image, label) in zip(axes.flat, train_ds.unbatch().take(6)):
    ax.imshow(tf.cast(image, tf.uint8).numpy())
    ax.set_title(class_names[int(label.numpy())]); ax.axis("off")
plt.tight_layout(); plt.show()


<h2 dir="rtl" style="direction:rtl;text-align:right">ما الذي تفعله دوال تجهيز البيانات؟</h2>
<ul dir="rtl" style="direction:rtl;text-align:right"><li><code>train_test_split(..., stratify=...)</code>: يحافظ تقريبًا على نسب الفئات في كل جزء.</li><li><code>load_image</code>: يقرأ الملف ويحوّله إلى RGB ويعيد تحجيمه إلى 160×160.</li><li><code>make_dataset</code>: يبني دفعات TensorFlow؛ يخلط التدريب فقط، ويستخدم prefetch لتجهيز الدفعة التالية.</li><li><code>compute_class_weight</code>: يعطي وزنًا أعلى للفئات الأقل تمثيلًا أثناء حساب الخسارة.</li></ul>
<p dir="rtl" style="direction:rtl;text-align:right">مجموعة الاختبار تبقى خارج عملية التدريب، ونستخدمها مرة واحدة للتقييم النهائي.</p>

<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">بناء CNN وتدريبها</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">نضيف انعكاسًا ودورانًا خفيفًا أثناء التدريب فقط. تختار EarlyStopping أفضل مرحلة اعتمادًا على مجموعة التحقق. غيّر عدد الحقب بعد أن تتأكد أن الدفتر يعمل على Colab.</p>

<h2 dir="rtl" style="direction:rtl;text-align:right">قراءة بنية CNN قبل التدريب</h2>
<p dir="rtl" style="direction:rtl;text-align:right">طبقات الالتفاف تستخرج أنماطًا محلية مثل الحواف والملمس، وMaxPooling يقلل الأبعاد المكانية. ثم GlobalAveragePooling يلخّص خرائط الخصائص، وDense يربطها بالفئات. طبقة Softmax تعطي احتمالات مجموعها 1. الرسم التالي تمثيل تعليمي لأحجام الموترات، وليس تصويرًا للمعالج أو نشاطًا زمنيًا حقيقيًا.</p>

In [ ]:
# محاكاة ثلاثية الأبعاد توضيحية لأحجام الموترات عبر طبقات الشبكة.
# القيم z تعبر عن عدد القنوات/المرشحات بصريًا؛ ليس قياسًا ماديًا.
from plotly.subplots import make_subplots

def show_cnn_tensor_blocks():
    stages = [(160,160,3,"Input"),(160,160,32,"Conv 32"),(80,80,32,"Pool"),(80,80,64,"Conv 64"),(40,40,64,"Pool"),(40,40,128,"Conv 128"),(20,20,128,"Pool"),(1,1,6,"Classes")]
    fig = go.Figure()
    for i,(h,w,ch,name) in enumerate(stages):
        # Cuboids encode H,W,C as normalized dimensions for readable comparison.
        dx,dy,dz = max(w/160,.08), max(h/160,.08), max(ch/128,.05)
        x0=i*1.45; y0=0; z0=0
        verts=[(x0,y0,z0),(x0+dx,y0,z0),(x0+dx,y0+dy,z0),(x0,y0+dy,z0),(x0,y0,z0+dz),(x0+dx,y0,z0+dz),(x0+dx,y0+dy,z0+dz),(x0,y0+dy,z0+dz)]
        faces=[(0,1,2),(0,2,3),(4,5,6),(4,6,7),(0,1,5),(0,5,4),(1,2,6),(1,6,5),(2,3,7),(2,7,6),(3,0,4),(3,4,7)]
        xx,yy,zz=zip(*verts); ii,jj,kk=zip(*faces)
        fig.add_trace(go.Mesh3d(x=xx,y=yy,z=zz,i=ii,j=jj,k=kk,opacity=.82,color=["#4cc9f0","#4895ef","#4361ee","#3f37c9","#7209b7","#b5179e","#f72585","#ffb703"][i],name=f"{name}: {h}×{w}×{ch}",hovertemplate=f"{name}<br>shape={h}×{w}×{ch}<extra></extra>"))
        fig.add_trace(go.Scatter3d(x=[x0+dx/2],y=[y0+dy/2],z=[dz+.08],mode="text",text=[name],showlegend=False,hoverinfo="skip"))
    fig.update_layout(title="CNN tensor flow — conceptual 3D view",scene=dict(xaxis_title="Layer order",yaxis_title="Spatial width (scaled)",zaxis_title="Channels (scaled)", camera=dict(eye=dict(x=1.6,y=1.8,z=1.4))),height=560,showlegend=True)
    fig.show()

show_cnn_tensor_blocks()

In [ ]:
# إنشاء النموذج عبر الدالة المشتركة المستخدمة كذلك في تطبيق الويب.
model = wsh.build_cnn(
    tf=tf,
    class_count=len(class_names),
    image_size=IMAGE_SIZE,
    filters=32,
    dropout=0.35,
    learning_rate=0.001,
    augmentation=True,
)
model.summary()


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=12,
                    class_weight=class_weights, callbacks=callbacks)


<h2 dir="rtl" style="direction:rtl;text-align:right">محاكاة خرائط الخصائص التي تعلمتها الشبكة</h2>
<p dir="rtl" style="direction:rtl;text-align:right">نأخذ قناة واحدة من مخرجات طبقات الالتفاف لصورة اختبار ونرسم قيمها كسطح ثلاثي الأبعاد. كل قناة استجابة مرشح مختلف؛ الألوان والارتفاعات تمثل شدة الاستجابة. هذا يوضح أين تستجيب الطبقة، ولا يثبت وحده سبب قرار النموذج.</p>

In [ ]:
# دالة مساعدة: إظهار استجابة قناة من كل طبقة Conv2D لعينة اختبار.
def show_feature_surfaces(dataset, sample_index=0, channels=(0,0,0), max_side=48):
    image_batch, label_batch = next(iter(dataset.skip(sample_index).take(1)))
    conv_layers = [layer for layer in model.layers if isinstance(layer, tf.keras.layers.Conv2D)]
    feature_model = tf.keras.Model(model.input, [layer.output for layer in conv_layers])
    activations = feature_model(image_batch[:1], training=False)
    fig = make_subplots(rows=1, cols=len(activations), specs=[[{"type":"surface"}]*len(activations)], subplot_titles=[f"Conv {i+1}" for i in range(len(activations))])
    for col, (act, channel) in enumerate(zip(activations, channels), 1):
        plane = act[0,:,:,channel].numpy()
        plane = cv2.resize(plane, (min(max_side,plane.shape[1]), min(max_side,plane.shape[0])))
        fig.add_trace(go.Surface(z=plane, colorscale="Turbo", showscale=False), row=1, col=col)
    fig.update_layout(title=f"Feature maps — true label: {class_names[int(label_batch[0])]}",height=480)
    fig.show()

show_feature_surfaces(test_ds)

<h2 dir="rtl" style="direction:rtl;text-align:right">كيف نقرأ مخطط التدريب؟</h2>
<p dir="rtl" style="direction:rtl;text-align:right"><strong>الخسارة:</strong> الأقل أفضل على البيانات نفسها. إذا انخفضت خسارة التدريب بينما بقيت خسارة التحقق أعلى أو ارتفعت، فقد يكون هناك فرط تعلّم. <strong>الدقة:</strong> نسبة التوقعات الصحيحة، لكنها قد تخفي ضعف الفئات النادرة؛ لذلك نراجع Precision وRecall وF1 ومصفوفة الالتباس أيضًا. Epoch تعني مرورًا كاملًا على بيانات التدريب.</p>

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["loss"], label=rtl_text("تدريب")); ax[0].plot(history.history["val_loss"], label=rtl_text("تحقق"))
ax[0].set_title(rtl_text("الخسارة")); ax[0].set_xlabel("Epoch"); ax[0].legend(); ax[0].grid(alpha=.2)
ax[1].plot(history.history["accuracy"], label=rtl_text("تدريب")); ax[1].plot(history.history["val_accuracy"], label=rtl_text("تحقق"))
ax[1].set_title(rtl_text("الدقة")); ax[1].set_xlabel("Epoch"); ax[1].legend(); ax[1].grid(alpha=.2)
plt.tight_layout(); plt.show()


<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تقييم الاختبار</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">نستخدم الاختبار بعد التدريب فقط. راجع أداء كل فئة خصوصًا الفئة الأقل عددًا، ولا تعتمد على Accuracy وحدها.</p>

<h2 dir="rtl" style="direction:rtl;text-align:right">المقاييس ومصفوفة الالتباس</h2>
<p dir="rtl" style="direction:rtl;text-align:right">كل صف في المصفوفة هو الفئة الحقيقية وكل عمود هو توقع النموذج. القطر الرئيسي توقع صحيح؛ القيم خارج القطر تبين الالتباس. Precision يجيب: من الصور التي توقعها النموذج لهذه الفئة، كم كان صحيحًا؟ Recall يجيب: من صور هذه الفئة الحقيقية، كم اكتشف؟ وF1 يوازن بينهما.</p>

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
probs = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)
print("دقة الاختبار:", round(accuracy_score(test_y, y_pred), 3))
print(classification_report(test_y, y_pred, target_names=class_names, digits=3, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(test_y, y_pred), display_labels=class_names).plot(
    cmap="Blues", xticks_rotation=35, values_format="d", colorbar=False)
plt.title(rtl_text("مصفوفة الالتباس — الاختبار")); plt.tight_layout(); plt.show()


<h2 dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">تجربة صورة جديدة من جهازك</h2>
<p dir="rtl" style="direction:rtl;text-align:right;unicode-bidi:plaintext">في Google Colab شغّل الخلية وارفع صورة واحدة. يطبع النموذج الفئة المتوقعة واحتمالها. جرّب صورًا ذات خلفية وإضاءة مختلفتين عن خلفية المجموعة، وسجّل الحالات التي أخطأ فيها.</p>

<h2 dir="rtl" style="direction:rtl;text-align:right">واجهة التجربة التفاعلية</h2>
<p dir="rtl" style="direction:rtl;text-align:right">بعد التدريب تظهر واجهة لاختيار صورة، عرض معاينة، تشغيل التصنيف، واستعراض احتمالات الفئات مع خيار CLAHE. إذا ظهرت رسالة أن أداة الرفع تحتاج تنفيذ الخلية في جلسة المتصفح الحالية، أعد تشغيل خلية الواجهة بعد اتصال Colab؛ فالودجت لا يُستعاد تلقائيًا بعد انقطاع الجلسة. مرجع تفعيل مدير الودجات في Colab: <a href="https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_widgets.py">Colab output widgets</a>.</p>

In [ ]:
from google.colab import output
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from PIL import Image as PILImage
import io, html

# تفعيل عرض الواجهات التفاعلية داخل Google Colab.
output.enable_custom_widget_manager()

names_ar = {
    "cardboard": "كرتون", "glass": "زجاج", "metal": "معدن",
    "paper": "ورق", "plastic": "بلاستيك", "trash": "نفايات أخرى"
}

def get_upload_record(value):
    if not value:
        return None, None
    if isinstance(value, dict):  # ipywidgets 7
        name, record = next(iter(value.items()))
    else:  # ipywidgets 8
        record = value[0]
        name = record.get("name", "image") if isinstance(record, dict) else record.name
    content = record.get("content") if isinstance(record, dict) else record.content
    return name, bytes(content)

def make_preview(rgb):
    h, w = rgb.shape[:2]
    scale = min(260 / w, 210 / h, 1.0)
    small = cv2.resize(rgb, (max(1, int(w * scale)), max(1, int(h * scale))))
    stream = io.BytesIO()
    PILImage.fromarray(small).save(stream, format="PNG")
    return stream.getvalue()

uploader = widgets.FileUpload(accept="image/*", multiple=False, description="اختر صورة")
analyze_button = widgets.Button(description="حلّل الصورة", button_style="success", icon="search")
show_clahe = widgets.Checkbox(value=False, description="عرض تحسين التباين CLAHE")
result_box = widgets.Output()

header = widgets.HTML(value="""
<div dir="rtl" style="direction:rtl;text-align:right;background:linear-gradient(120deg,#063b46,#087f8c);color:white;padding:18px 22px;border-radius:14px;font-family:Arial,sans-serif">
  <div style="font-size:22px;font-weight:700">♻️ مساعد الفرز الذكي</div>
  <div style="margin-top:6px;font-size:14px">ارفع صورة نفاية، وشاهد تصنيف CNN وفحص الصورة بـOpenCV</div>
</div>""")
hint = widgets.HTML(value="<div dir='rtl' style='direction:rtl;text-align:right;color:#64748b'>التصنيف تعليمي فقط. درجة الثقة ليست ضمانًا لصحة الفرز.</div>")

def on_analyze(_):
    with result_box:
        clear_output(wait=True)
        filename, content = get_upload_record(uploader.value)
        if content is None:
            display(HTML("<div dir='rtl' style='direction:rtl;text-align:right;color:#b45309'>اختر صورة أولًا.</div>"))
            return
        try:
            original = np.asarray(PILImage.open(io.BytesIO(content)).convert("RGB"))
        except Exception as exc:
            display(HTML(f"<div dir='rtl' style='direction:rtl;text-align:right;color:#b91c1c'>تعذر قراءة الصورة: {html.escape(str(exc))}</div>"))
            return

        # OpenCV: تحويل RGB إلى LAB وتحسين قناة الإضاءة محليًا بـ CLAHE.
        bgr = cv2.cvtColor(original, cv2.COLOR_RGB2BGR)
        lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab)
        enhanced_l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l_channel)
        enhanced_rgb = cv2.cvtColor(cv2.cvtColor(cv2.merge([enhanced_l, a_channel, b_channel]), cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)

        # النموذج تدرب على RGB الأصلية؛ CLAHE للعرض والمقارنة فقط.
        model_input = cv2.resize(original, IMG_SIZE)
        scores = model.predict(model_input[None, ...], verbose=0)[0]
        top_indices = np.argsort(scores)[::-1][:3]
        best = int(top_indices[0])
        raw_name = class_names[best]
        arabic_name = names_ar.get(raw_name.lower(), raw_name)
        confidence = float(scores[best])
        confidence_color = "#16a34a" if confidence >= .75 else "#d97706" if confidence >= .50 else "#dc2626"

        bars = "".join(
            f"<div style='margin:9px 0'><div style='display:flex;justify-content:space-between;direction:rtl'><span>{html.escape(names_ar.get(class_names[int(i)].lower(), class_names[int(i)]))}</span><b>{scores[int(i)]:.1%}</b></div>"
            f"<div style='height:8px;background:#e2e8f0;border-radius:8px;overflow:hidden;margin-top:4px'><div style='width:{scores[int(i)]*100:.1f}%;height:100%;background:#0e9aa7;border-radius:8px'></div></div></div>"
            for i in top_indices
        )
        display(HTML(f"""
        <div dir="rtl" style="direction:rtl;text-align:right;font-family:Arial,sans-serif;background:#f8fafc;border:1px solid #dbe4ea;padding:16px 20px;border-radius:14px;margin:8px 0">
          <div style="font-size:13px;color:#64748b">نتيجة تحليل الصورة: {html.escape(filename)}</div>
          <div style="font-size:24px;font-weight:700;color:#0f172a;margin-top:5px">الفئة: {html.escape(arabic_name)}</div>
          <div style="font-size:16px;color:{confidence_color};margin-top:4px">ثقة النموذج: {confidence:.1%}</div>
          <div style="margin-top:12px;font-weight:600">أعلى التوقعات</div>{bars}
        </div>"""))

        previews = [widgets.VBox([widgets.HTML("<div dir='rtl' style='direction:rtl;text-align:center;font-weight:600'>الصورة الأصلية</div>"), widgets.Image(value=make_preview(original), format="png")])]
        if show_clahe.value:
            previews.append(widgets.VBox([widgets.HTML("<div dir='rtl' style='direction:rtl;text-align:center;font-weight:600'>بعد تحسين CLAHE</div>"), widgets.Image(value=make_preview(enhanced_rgb), format="png")]))
        display(widgets.HBox(previews))

analyze_button.on_click(on_analyze)
display(widgets.VBox([
    header,
    widgets.HBox([uploader, analyze_button, show_clahe], layout=widgets.Layout(flex_flow="row wrap", align_items="center", gap="10px")),
    hint,
    result_box,
], layout=widgets.Layout(width="100%")))

<h2 dir="rtl" style="direction:rtl;text-align:right">خلاصة المشروع والتقرير</h2>
<p dir="rtl" style="direction:rtl;text-align:right">بعد التشغيل، دوّن دقة الاختبار، أكثر زوج فئات التبس، وما إذا اختلفت النتائج على صور الهاتف. لا تنس حفظ النموذج والرسوم. تقرير Word المرافق يشرح مراحل العمل ومخطط كل رسم، ويعرض نتائج التشغيل السابقة المرسلة بصفتها قراءة من لقطة شاشة، لا نتيجة مضمونة للنسخة الحالية.</p>
<p dir="rtl" style="direction:rtl;text-align:right"><strong>حدود التجربة:</strong> TrashNet صغيرة وصورها غالبًا على خلفية بيضاء؛ فئة trash لا تعني كل النفايات العضوية، وقد لا يعمم النموذج على بيئة مختلفة. استخدم صور اختبار جديدة وخلفيات وإضاءات متنوعة قبل أي استنتاج عملي.</p>
<p dir="rtl" style="direction:rtl;text-align:right"><strong>حفظ النموذج:</strong> <code dir="ltr" style="direction:ltr">model.save("waste_classifier.keras")</code></p>
<p dir="rtl" style="direction:rtl;text-align:right"><strong>المصدر:</strong> Thung &amp; Yang, Stanford CS229 (2016), <a href="https://github.com/garythung/trashnet">TrashNet</a>.</p>